<a href="https://www.kaggle.com/code/briankimanzi/bean-classification?scriptVersionId=307323387" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Bean Classification - YOLO Segmentation
**Kaggle Notebook** | YOLOv8 Instance Segmentation for Bean Classification

In [35]:
!pip install ultralytics -q
!pip install pandas numpy Pillow opencv-python-headless -q

In [36]:
import os, shutil, ast, random, cv2
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
import torch
from sklearn.model_selection import train_test_split
from IPython.display import FileLink, display
import yaml

In [37]:
import os

BASE_DIR      = '/kaggle/input/Beans Classification'  # adjust to your dataset name
TRAIN_CSV     = f'{BASE_DIR}/Train.csv'
TEST_CSV      = f'{BASE_DIR}/Test.csv'
TRAIN_IMG_DIR = f'{BASE_DIR}/images'
TEST_IMG_DIR  = f'{BASE_DIR}/images'
YOLO_DIR      = '/kaggle/working/yolo_dataset'

print(f'Train CSV: {TRAIN_CSV}')
print(f'Test  CSV: {TEST_CSV}')

Train CSV: /kaggle/input/Beans Classification/Train.csv
Test  CSV: /kaggle/input/Beans Classification/Test.csv


In [38]:
# Auto-discover dataset paths from Kaggle input
import os

print("Available input files:")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


Available input files:
/kaggle/input/datasets/briankimanzi/beans-classification-testing/Test.csv
/kaggle/input/datasets/briankimanzi/beans-clasification-training/Train.csv


In [39]:
import gdown, zipfile, os
from pathlib import Path

# Paste your Google Drive file ID here (the long string from the share link)
FILE_ID = "1WJPkaa8lNhE13eVb_mjhcdeano_7tH8R"  # e.g. "1aBcDeFgHiJkLmNoPqRsTuVwXyZ"
# https://drive.google.com/file/d//view?usp=sharing

# Download
print("Downloading images.zip from Google Drive...")
gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", "/kaggle/working/images.zip", quiet=False)

# Extract
print("Extracting...")
with zipfile.ZipFile("/kaggle/working/images.zip", "r") as z:
    z.extractall("/kaggle/working/")

print("Done! Checking extracted contents:")
for dirname, dirs, filenames in os.walk('/kaggle/working'):
    if filenames:
        print(f"📁 {dirname}  ({len(filenames)} files)")

Downloading...
From (original): https://drive.google.com/uc?id=1WJPkaa8lNhE13eVb_mjhcdeano_7tH8R
From (redirected): https://drive.google.com/uc?id=1WJPkaa8lNhE13eVb_mjhcdeano_7tH8R&confirm=t&uuid=811a2aac-4962-45bf-b2ac-be3d0387a978
To: /kaggle/working/images.zip
100%|██████████| 6.07G/6.07G [00:42<00:00, 142MB/s] 


Extracting...
Done! Checking extracted contents:
📁 /kaggle/working  (6 files)
📁 /kaggle/working/images  (2713 files)
📁 /kaggle/working/runs/segment/val2  (16 files)
📁 /kaggle/working/runs/segment/val  (16 files)
📁 /kaggle/working/runs/bean_seg  (26 files)
📁 /kaggle/working/runs/bean_seg/weights  (2 files)
📁 /kaggle/working/yolo_dataset  (1 files)
📁 /kaggle/working/yolo_dataset/images/train  (837 files)
📁 /kaggle/working/yolo_dataset/images/val  (148 files)
📁 /kaggle/working/yolo_dataset/labels  (2 files)
📁 /kaggle/working/yolo_dataset/labels/train  (837 files)
📁 /kaggle/working/yolo_dataset/labels/val  (148 files)
📁 /kaggle/working/.virtual_documents  (1 files)


In [40]:
TRAIN_CSV     = '/kaggle/input/datasets/briankimanzi/beans-clasification-training/Train.csv'
TEST_CSV      = '/kaggle/input/datasets/briankimanzi/beans-classification-testing/Test.csv'
TRAIN_IMG_DIR = '/kaggle/working/images'   # adjust if zip extracts to a different folder name
TEST_IMG_DIR  = '/kaggle/working/images'
YOLO_DIR      = '/kaggle/working/yolo_dataset'

In [41]:
import os
for dirname, dirs, filenames in os.walk('/kaggle/input'):
    print(f"📁 {dirname}  ({len(filenames)} files)")


📁 /kaggle/input  (0 files)
📁 /kaggle/input/datasets  (0 files)
📁 /kaggle/input/datasets/briankimanzi  (0 files)
📁 /kaggle/input/datasets/briankimanzi/beans-classification-testing  (1 files)
📁 /kaggle/input/datasets/briankimanzi/beans-clasification-training  (1 files)


In [43]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print()
print("Train columns:", train_df.columns.tolist())
print()
print("Class distribution:")
print(train_df['class'].value_counts())
print()
print("Unique train images:", train_df["Image_ID"].nunique())
print("Unique test images:", test_df["Image_ID"].nunique())


Train shape: (12142, 6)
Test shape: (352, 3)

Train columns: ['Image_ID', 'image_width', 'image_height', 'bbox', 'points', 'class']

Class distribution:
class
Flower_closed    6003
Flower_open      4085
Plant_bean       2054
Name: count, dtype: int64

Unique train images: 985
Unique test images: 352


In [44]:
import os, shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

CLASS_MAP = {
    'Plant_bean':    0,
    'Flower_open':   1,
    'Flower_closed': 2
}
ID_TO_CLASS = {v: k for k, v in CLASS_MAP.items()}

def parse_points(points_str):
    """Parse 'x1,y1;x2,y2;...' into list of (x, y) tuples."""
    pairs = points_str.strip().split(';')
    coords = []
    for p in pairs:
        x, y = p.strip().split(',')
        coords.append((float(x), float(y)))
    return coords

def normalize_polygon(points, img_w, img_h):
    """Normalize polygon coordinates to [0, 1] for YOLO format."""
    normalized = []
    for x, y in points:
        normalized.extend([x / img_w, y / img_h])
    return normalized

def create_yolo_dataset(df, img_src_dir, split_name, yolo_dir):
    img_out = Path(yolo_dir) / 'images' / split_name
    lbl_out = Path(yolo_dir) / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    grouped = df.groupby('Image_ID')
    skipped = 0

    for img_id, group in grouped:
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            candidate = Path(img_src_dir) / f'{img_id}{ext}'
            if candidate.exists():
                img_path = candidate
                break

        if img_path is None:
            skipped += 1
            continue

        dest_img = img_out / img_path.name
        if not dest_img.exists():
            shutil.copy(img_path, dest_img)

        img_w = group['image_width'].iloc[0]
        img_h = group['image_height'].iloc[0]
        label_lines = []

        for _, row in group.iterrows():
            cls_id = CLASS_MAP.get(row['class'])
            if cls_id is None:
                continue
            try:
                points = parse_points(row['points'])
                norm = normalize_polygon(points, img_w, img_h)
                coords_str = ' '.join(f'{v:.6f}' for v in norm)
                label_lines.append(f'{cls_id} {coords_str}')
            except Exception as e:
                continue

        lbl_file = lbl_out / f'{img_id}.txt'
        with open(lbl_file, 'w') as f:
            f.write('\n'.join(label_lines))

    print(f'[{split_name}] Done. Skipped {skipped} images not found on disk.')
    return img_out, lbl_out

unique_ids = train_df['Image_ID'].unique()
train_ids, val_ids = train_test_split(unique_ids, test_size=0.15, random_state=42)

train_split = train_df[train_df['Image_ID'].isin(train_ids)]
val_split   = train_df[train_df['Image_ID'].isin(val_ids)]

print(f'Train images: {len(train_ids)} | Val images: {len(val_ids)}')

if os.path.exists(YOLO_DIR):
    shutil.rmtree(YOLO_DIR)

create_yolo_dataset(train_split, TRAIN_IMG_DIR, 'train', YOLO_DIR)
create_yolo_dataset(val_split,   TRAIN_IMG_DIR, 'val',   YOLO_DIR)

print('YOLO dataset ready')

Train images: 837 | Val images: 148
[train] Done. Skipped 0 images not found on disk.
[val] Done. Skipped 0 images not found on disk.
YOLO dataset ready


In [45]:
# Create YOLO data.yaml
data_yaml = {
    'path': YOLO_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 3,
    'names': ['Plant_bean', 'Flower_open', 'Flower_closed']
}

yaml_path = f'{YOLO_DIR}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)
print('data.yaml contents:')
print(open(yaml_path).read())

data.yaml contents:
names:
- Plant_bean
- Flower_open
- Flower_closed
nc: 3
path: /kaggle/working/yolo_dataset
train: images/train
val: images/val



In [ ]:
# Train YOLOV8-seg Model
model = YOLO('yolov8x-seg.pt')

device = 0 if torch.cuda.is_available() else 'cpu'
print(f'Using device: {"GPU (cuda:0)" if device == 0 else "CPU"}')

if device == 'cpu':
    print('WARNING: Training on CPU will be very slow.')
    print('Go to Runtime → Change runtime type → T4 GPU')
# Train
results = model.train(
    data    = yaml_path,
    epochs  =150,
    imgsz   = 1024,
    batch   = 4,
    patience= 30,
    device  = device,
    project = '/kaggle/working/runs',
    name = 'bean_seg_v2',
    exist_ok = True,
    optimizer = 'AdamW',
    lr0          = 0.001,
    lrf          = 0.01,
    weight_decay = 0.0005,
    warmup_epochs= 3,
# Augmentation
    hsv_h = .015,
    hsv_s = .7,
    hsv_v = .4,
    flipud = .5,
    fliplr = .5,
    mosaic = 1.0,
    degrees = 15.0,
    translate = .1,
    scale = .5,
    copy_paste=.3,
    mixup=.15,
    shear=2.0,
)
print("Training complete")
print("Best Model Saved at:", results.save_dir)

Using device: GPU (cuda:0)
Ultralytics 8.4.31 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8x-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bean_seg_v2, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam

In [ ]:
results = model.predict(
    source  = img_path,
    imgsz   = 1024,    
    conf    = 0.25,
    iou     = 0.45,
    augment = True,   
    verbose = False
)

In [ ]:
import os
from IPython.display import FileLink, display

In [ ]:
import shutil

# Copy to Kaggle output directory — this makes it appear in the Output tab
shutil.copy('/kaggle/working/runs/bean_seg/weights/best.pt', '/kaggle/working/best.pt')
print("Done! Now go to:")
print("  Right sidebar → Output tab → best.pt → Download")

In [ ]:
# Evaluate on Validation set
best_model_path ="/kaggle/working/runs/bean_seg/weights/best.pt"
model = YOLO(best_model_path)

# Validate
metrics = model.val(data=yaml_path, imgsz=640, iou=.50, conf=.25)
print("\n===== Validation metrics ===")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAp50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
# Generate Predictions on Test set
CONF_THRESHOLD =.25
IOU_THRESHOLD = .45

CLASS_NAMES = ["Plant_bean", "Flower_open", "Flower_closed"]

def get_test_image_path(img_id, img_dir):
    for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
        p = Path(img_dir) / f"{img_id}{ext}"
        if  p.exists():
            return str(p)
    return None

submission_rows = []
test_ids = test_df["Image_ID"].unique()
missing = []

print(f"Running inference on {len(test_ids)} test Iimages...")

for i, img_id in enumerate(test_ids):
    img_path = get_test_image_path(img_id, TEST_IMG_DIR)

    if img_path is None:
        missing.append(img_id)
        submission_rows.append({
            'Image_ID': img_id,
            'class': '__background__',
            'confidence': .0,
            'ymin': 0, 'xmin': 0, 'ymax': 0, 'xmax':0
        })
        continue

    results = model.predict(
        source = img_path,
        imgsz = 640,
        conf = CONF_THRESHOLD,
        iou = IOU_THRESHOLD,
        verbose = False
    )

    result = results[0]
    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        submission_rows.append({
            'Image_ID':   img_id,
            'class':      '__background__',
            'confidence': 0.0,
            'ymin': 0, 'xmin': 0, 'ymax': 0, 'xmax': 0
        })
        continue
    orig_h, orig_w = result.orig_shape

    for j in range(len(boxes)):
        conf = float(boxes.conf[j].cpu())
        cls_id = int(boxes.cls[j].cpu())
        cls_name = CLASS_NAMES[cls_id]

        if result.masks is not None and j < len(result.masks):
            mask = result.masks.data[j].cpu().numpy()
            mask_resized = cv2.resize(
                mask, (orig_w, orig_h), interpolation =cv2.INTER_NEAREST
            )
            ys, xs = np.where(mask_resized > .5)
            if len(xs) > 0 and len(ys) > 0:
                xmin, xmax = int(xs.min()), int(xs.max())
                ymin, ymax = int(ys.min()), int(ys.max())
            else:
                x1, y1, x2,y2 = boxes.xyxy[j].cpu().numpy()
                xmin, ymin, xmax, ymax = int(x1), int(y1), int(x2), int(y2)

        else:
            x1, y1, x2, y2 = boxes.xyxy[j].cpu().numpy()
            xmin, ymin, xmax, ymax = int(x1), int(y1), int(x2), int(y2)

        submission_rows.append({
            'Image_ID': img_id,
            'class': cls_name,
            'confidence': round(conf, 6),
            'ymin': ymin,
            'xmin': xmin,
            'ymax': ymax,
            'xmax': xmax
        })

    if (i + 1) % 50 == 0:
        print(f" {i +1}/{len(test_ids)} processed ...")
print(f"\n Inference done. Missing images: {len(missing)}")

In [ ]:
# Bulding and saving the submission csv
submission_df = pd.DataFrame(submission_rows, columns=[
    'Image_ID', 'class', 'confidence', 'ymin', 'xmin', 'ymax', 'xmax'
])
submission_df = submission_df.sort_values(
    ['Image_ID', "confidence"], ascending=[True, False]
).reset_index(drop=True)

print("Submission Sumaary")
print(f"Total rows: {len(submission_df)}")
print(f"Unique Image_IDs:   {submission_df["Image_ID"].nunique()}")
print(f"Expected images:    {len(test_ids)}")
print()
print('class distribution in predictions:')
print(submission_df['class'].value_counts())
print()
print('Sample rows: ')
display(submission_df.head(10))

submitted_ids = set(submission_df['Image_ID'].unique())
missing_from_submission = set(test_ids) - submitted_ids
if missing_from_submission:
    print(f"\n error: {len(missing_from_submission)} images missing from submission!")

    bg_rows = [{
        'Image_ID': img_id, 'class': "__background__",
        'confidence': .0, 'ymin': 0, 'xmin':0, 'ymax':0, "xmax":0
    } for img_id in missing_from_submission]
    sumission_df = pd.concat(
        [submission_df, pd.DataFrame(bg_rows)], ignore_index=True
    )
    print("Background rows added for missing images")
else:
    print("All test images are covered in submission!!")

SUBMISSION_PATH = "/kaggle/working/submission.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Submission saved to: {SUBMISSION_PATH}")

In [ ]:
import os, pandas as pd, shutil
from IPython.display import FileLink, display

SUBMISSION = '/kaggle/working/submission.csv'

# Confirm it exists
print(f"File exists: {os.path.exists(SUBMISSION)}")
print(f"Size: {os.path.getsize(SUBMISSION) / 1024:.1f} KB")

sub = pd.read_csv(SUBMISSION)
print(f"Rows: {len(sub)}")
print()
print(sub.head())

# Copy to output so it shows in the Output tab
shutil.copy(SUBMISSION, '/kaggle/working/submission_download.csv')
print("\nGo to: Right sidebar → Output tab → submission_download.csv → Download")

In [ ]:
# This estimates F1 at different confidence thresholds using your val split
# so you can pick the best one before final submission

from collections import defaultdict

def compute_iou_bbox(boxA, boxB):
    """Compute IoU between two [ymin, xmin, ymax, xmax] boxes."""
    yA = max(boxA[0], boxB[0]); xA = max(boxA[1], boxB[1])
    yB = min(boxA[2], boxB[2]); xB = min(boxA[3], boxB[3])
    inter = max(0, yB - yA) * max(0, xB - xA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

def f1_at_iou50(preds, gts, iou_thresh=0.50):
    """Compute F1 given list of pred boxes and gt boxes."""
    tp = fp = fn = 0
    matched = set()
    for pred in preds:
        best_iou, best_idx = 0, -1
        for idx, gt in enumerate(gts):
            if idx in matched:
                continue
            if pred['class'] != gt['class']:
                continue
            iou = compute_iou_bbox(
                [pred['ymin'], pred['xmin'], pred['ymax'], pred['xmax']],
                [gt['ymin'],   gt['xmin'],   gt['ymax'],   gt['xmax']]
            )
            if iou > best_iou:
                best_iou, best_idx = iou, idx
        if best_iou >= iou_thresh:
            tp += 1
            matched.add(best_idx)
        else:
            fp += 1
    fn = len(gts) - len(matched)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return f1, precision, recall

print('Running confidence threshold sweep on validation set...')
print('(This may take a few minutes depending on val set size)')

# Build ground truth from val split
gt_by_image = defaultdict(list)
for _, row in val_split.iterrows():
    try:
        bbox = row['bbox'].split(',')
        x, y, w, h = float(bbox[0]), float(bbox[1]), float(bbox[2]), float(bbox[3])
        gt_by_image[row['Image_ID']].append({
            'class': row['class'],
            'ymin': y, 'xmin': x, 'ymax': y + h, 'xmax': x + w
        })
    except:
        continue

# Sweep thresholds
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
results_sweep = []

for conf_thresh in thresholds:
    all_preds = []
    all_gts   = []

    for img_id in list(gt_by_image.keys())[:50]:  # use first 50 val images for speed
        img_path = get_test_image_path(img_id, TRAIN_IMG_DIR)
        if img_path is None:
            continue
        preds_result = model.predict(
            source=img_path, imgsz=640,
            conf=conf_thresh, iou=IOU_THRESHOLD, verbose=False
        )[0]
        preds = []
        if preds_result.boxes is not None:
            for j in range(len(preds_result.boxes)):
                x1,y1,x2,y2 = preds_result.boxes.xyxy[j].cpu().numpy()
                preds.append({
                    'class': CLASS_NAMES[int(preds_result.boxes.cls[j])],
                    'ymin': y1, 'xmin': x1, 'ymax': y2, 'xmax': x2
                })
        all_preds.extend(preds)
        all_gts.extend(gt_by_image[img_id])

    f1, prec, rec = f1_at_iou50(all_preds, all_gts)
    results_sweep.append({'conf': conf_thresh, 'F1': f1, 'Precision': prec, 'Recall': rec})
    print(f'conf={conf_thresh:.2f}  F1={f1:.4f}  Prec={prec:.4f}  Rec={rec:.4f}')

sweep_df = pd.DataFrame(results_sweep)
best_row = sweep_df.loc[sweep_df['F1'].idxmax()]
print(f'\nBest confidence threshold: {best_row["conf"]} → F1={best_row["F1"]:.4f}')
print('→ Re-run Step 8 with CONF_THRESHOLD =', best_row['conf'])

In [ ]:
# # Tune Confidence Threshold for better F!
# from collections import defaultdict

# def computer_iou_bbox(boxA, boxB):
#     """
#     Compute IoU between two [ymin, xmin, ymax, xmax] boxes
#     """
#     yA = max(boxA[0], boxB[0]); xA = max(boxA[1], boxB[1])
#     yB = min(boxA[2], box[2]); xB = min(boxA[3], boxB[3])
#     inter = max(0, yB - yA) * max(0, xB - xA)
#     areaA = (boxA[2]-boxA[0]) * (boxA[3] - boxA[1])
#     areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
#     union = areaA + areaB - inter
#     return inter / union if union > 0 else .0

# def f1_at_

In [ ]:
# Data observation
# Show case the dataset information
def data_info(x):
    """
    This function is used to display the dataset information like:
    - shape of dataset
    - Statistical Info
    - Column Datatype
    - Null Value sum
    - Dulplicate values
    ** Parameters: X representing the data

    ** Return: None
    """
    print("================================================")
    print("Number of Rows and Columns In the DataSet")
    row, col = x.shape
    print(f"Data_Rows: {row} Data_col: {col}")
    print()
    print("================================================")
    print("Dataset Columns")
    print(x.columns)
    print()
    print("================================================")
    print("Dataset types")
    print(x.dtypes)
    print()
    print("================================================")
    print("DataSet Stats Info")
    print(x.describe())
    print()
    print("================================================")
    print("Dataset Column's Datatype")
    print(x.info())
    print()
    print("================================================")
    print("Dataset Null Count")
    print(x.isna().sum())
    print()
    print("================================================")
    print("Dataset Duplicated Values")
    print(x.duplicated().unique())
    print()
    print("================================================")
    print("Display All the Missing Values In a Chart")
    print(msno.bar(x))

    return None

In [ ]:
data_info(df_train)

In [ ]:
df_train.head(5)

In [ ]:
df_train["class"].unique()